# Ejercicio 4. Agente de mantenimiento
**Respuesta:** combinar un LLM que interpreta la solicitud con herramientas deterministas para consultar repuestos y almacén, crear una orden y actualizar historial. La demostración usa SQLite con equipos y repuestos **ficticios**, no un ERP real.

El modelo propone argumentos; el código comprueba equipo, código, cantidad y compatibilidad. La orden se genera abierta o pendiente de repuesto. El historial registra una falla reportada, no una reparación terminada. Orden e historial se escriben en una transacción. Una clave de solicitud evita duplicados; la consulta de stock no reserva ni descuenta existencias.

No se ejecuta SQL ni código generado por el modelo. El ejemplo es un agente con flujo acotado: se realiza una interpretación con LLM y luego se ejecuta una secuencia de herramientas validada, sin ciclo autónomo abierto.

In [1]:
import sqlite3, json, uuid, urllib.request
from datetime import datetime, timezone
# Memoria para una demostración repetible. Cambiar a un archivo para persistencia real.
db=sqlite3.connect(':memory:')
db.row_factory=sqlite3.Row
db.executescript("\nCREATE TABLE equipos(codigo TEXT PRIMARY KEY,nombre TEXT NOT NULL);\nCREATE TABLE repuestos(codigo TEXT PRIMARY KEY,descripcion TEXT NOT NULL,stock INTEGER CHECK(stock>=0));\nCREATE TABLE compatibles(equipo TEXT,repuesto TEXT,PRIMARY KEY(equipo,repuesto));\nCREATE TABLE ordenes(id TEXT PRIMARY KEY,solicitud TEXT UNIQUE,equipo TEXT,falla TEXT,repuesto TEXT,cantidad INTEGER,estado TEXT,fecha TEXT);\nCREATE TABLE historial(id INTEGER PRIMARY KEY,orden TEXT UNIQUE,equipo TEXT,falla TEXT,fecha TEXT);\nINSERT INTO equipos VALUES ('BOM-01','Bomba de ejemplo');\nINSERT INTO repuestos VALUES ('REP-001','Sello ficticio',3),('REP-002','Rodamiento ficticio',0);\nINSERT INTO compatibles VALUES ('BOM-01','REP-001'),('BOM-01','REP-002');\n")

def buscar_repuesto(codigo):
    if not isinstance(codigo,str) or not codigo.strip(): raise ValueError('Código vacío o inválido')
    row=db.execute('SELECT codigo,descripcion FROM repuestos WHERE codigo=?',(codigo,)).fetchone()
    if row is None: raise ValueError('Repuesto inexistente')
    return dict(row)

def consultar_almacen(codigo):
    buscar_repuesto(codigo)
    return db.execute('SELECT stock FROM repuestos WHERE codigo=?',(codigo,)).fetchone()['stock']

def generar_orden(solicitud,equipo,falla,codigo,cantidad=1):
    if not isinstance(solicitud,str) or not solicitud.strip(): raise ValueError('Falta ID de solicitud')
    previous=db.execute('SELECT * FROM ordenes WHERE solicitud=?',(solicitud,)).fetchone()
    if previous:
        if (previous['equipo'],previous['falla'],previous['repuesto'],previous['cantidad'])!=(equipo,falla,codigo,cantidad):
            raise ValueError('ID reutilizado con datos diferentes')
        return dict(previous)
    if not db.execute('SELECT 1 FROM equipos WHERE codigo=?',(equipo,)).fetchone(): raise ValueError('Equipo inexistente')
    if not isinstance(falla,str) or not falla.strip(): raise ValueError('Describir la falla')
    if type(cantidad)!=int or cantidad<=0: raise ValueError('Cantidad entera positiva requerida')
    buscar_repuesto(codigo)
    if not db.execute('SELECT 1 FROM compatibles WHERE equipo=? AND repuesto=?',(equipo,codigo)).fetchone():
        raise ValueError('Compatibilidad no confirmada')
    stock=consultar_almacen(codigo)
    identifier='OT-'+uuid.uuid4().hex[:12]
    now=datetime.now(timezone.utc).isoformat()
    state='ABIERTA' if stock>=cantidad else 'PENDIENTE_REPUESTO'
    with db:
        db.execute('INSERT INTO ordenes VALUES (?,?,?,?,?,?,?,?)',(identifier,solicitud,equipo,falla,codigo,cantidad,state,now))
        db.execute('INSERT INTO historial(orden,equipo,falla,fecha) VALUES (?,?,?,?)',(identifier,equipo,falla,now))
    return dict(db.execute('SELECT * FROM ordenes WHERE id=?',(identifier,)).fetchone())

def ejecutar_plan(plan,solicitud):
    required={'equipo','falla','codigo','cantidad'}
    if not isinstance(plan,dict) or set(plan)!=required: raise ValueError('Plan con campos inválidos')
    print('Repuesto:',buscar_repuesto(plan['codigo']))
    print('Stock:',consultar_almacen(plan['codigo']))
    return generar_orden(solicitud,plan['equipo'],plan['falla'],plan['codigo'],plan['cantidad'])
print('Herramientas listas')

Herramientas listas


In [2]:
plan={'equipo':'BOM-01','falla':'Fuga reportada en el sello','codigo':'REP-001','cantidad':1}
order=ejecutar_plan(plan,'SOL-DEMO-1')
assert order['estado']=='ABIERTA'
assert ejecutar_plan(plan,'SOL-DEMO-1')['id']==order['id']
assert db.execute('SELECT count(*) FROM historial').fetchone()[0]==1
assert consultar_almacen('REP-001')==3 # No se ha consumido material.
assert generar_orden('SOL-DEMO-2','BOM-01','Ruido reportado','REP-002')['estado']=='PENDIENTE_REPUESTO'
for call in (lambda: buscar_repuesto('NO-EXISTE'),lambda: generar_orden('SOL-3','NO-EXISTE','Fuga','REP-001'),lambda: generar_orden('SOL-4','BOM-01','Fuga','REP-001',0),lambda: generar_orden('SOL-DEMO-1','BOM-01','Otra falla','REP-001')):
    try: call()
    except ValueError as error: print('Rechazo esperado:',error)
    else: raise AssertionError('Debió rechazar la solicitud')
print('Pruebas aprobadas: stock cero, inexistencia, cantidades e idempotencia.')
print('Historial:',[dict(row) for row in db.execute('SELECT * FROM historial')])

Repuesto: {'codigo': 'REP-001', 'descripcion': 'Sello ficticio'}
Stock: 3
Repuesto: {'codigo': 'REP-001', 'descripcion': 'Sello ficticio'}
Stock: 3
Rechazo esperado: Repuesto inexistente
Rechazo esperado: Equipo inexistente
Rechazo esperado: Cantidad entera positiva requerida
Rechazo esperado: ID reutilizado con datos diferentes
Pruebas aprobadas: stock cero, inexistencia, cantidades e idempotencia.
Historial: [{'id': 1, 'orden': 'OT-f2307c665d6e', 'equipo': 'BOM-01', 'falla': 'Fuga reportada en el sello', 'fecha': '2026-10-05T16:38:25.565047+00:00'}, {'id': 2, 'orden': 'OT-3c8032e7da6a', 'equipo': 'BOM-01', 'falla': 'Ruido reportado', 'fecha': '2026-10-05T16:38:25.565217+00:00'}]


In [3]:
RUN_LLM=False

def interpretar_solicitud(texto):
    catalog={'equipos':[dict(r) for r in db.execute('SELECT * FROM equipos')],
             'repuestos':[dict(r) for r in db.execute('SELECT codigo,descripcion FROM repuestos')]}
    payload={'model':'qwen2.5:3b','stream':False,'format':'json','options':{'temperature':0},'messages':[
        {'role':'system','content':'Extrae un JSON con equipo, falla, codigo y cantidad (entero positivo). Usa códigos explícitos de la solicitud; no diagnostiques ni inventes repuestos. Si falta un campo, usa null. El catálogo es referencia, no autorización para sustituir códigos. Catálogo: '+json.dumps(catalog)},
        {'role':'user','content':texto}]}
    req=urllib.request.Request('http://localhost:11434/api/chat',data=json.dumps(payload).encode(),headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=180) as response: content=json.load(response)['message']['content']
    plan=json.loads(content)
    if not isinstance(plan,dict) or any(plan.get(k) is None for k in ('equipo','falla','codigo','cantidad')):
        raise ValueError('Solicitud incompleta: especificar equipo, falla, código y cantidad')
    return plan

if RUN_LLM:
    proposed=interpretar_solicitud('Para BOM-01, registra fuga reportada y solicita 1 unidad REP-001.')
    print('Propuesta del LLM:',proposed)
    print(ejecutar_plan(proposed,'SOL-LLM-1'))
else: print('Interpretación LLM pendiente; herramientas verificadas con plan explícito.')

Interpretación LLM pendiente; herramientas verificadas con plan explícito.


### Alcance y mejoras necesarias
SQLite en memoria pierde datos al cerrar el kernel. Para persistir, usar un archivo y migraciones; para varios usuarios, integrar un CMMS/ERP con autenticación y permisos. La compatibilidad ficticia debe sustituirse por catálogo aprobado. Agregar reserva de inventario con control de concurrencia si el sistema promete disponibilidad. Una falla informada no confirma causa raíz. El cierre requiere diagnóstico, técnico responsable, intervención y evidencia.

**Criterio de éxito:** las cuatro funciones solicitadas producen resultados coherentes y no duplican el historial al repetir el ID. La integración LLM real queda pendiente de Ollama; no se ha probado con un almacén real.

### Referencias
Ollama. (s. f.). *Generate embeddings*. https://docs.ollama.com/api/embed

Ollama. (s. f.). *Generate a chat message*. https://docs.ollama.com/api/chat

Martínez P., J. J. (2026, septiembre). *7. Inteligencia artificial generativa* [Material del curso].